In [ ]:
"""
Author: Hanan Abdulwahab Siala
University: King's College London
Date: 17-02-2026

Description:
    This program fine-tunes on one GPU to construct UML class diagrams from both Java and Python programs.
"""

In [ ]:
!nvidia-smi

In [ ]:
import pandas as pd
import torch

from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, GPTQConfig, TrainingArguments, BitsAndBytesConfig, Trainer, GenerationConfig
from transformers.models.mistral.modeling_mistral import MistralRotaryEmbedding
from peft import LoraConfig, AutoPeftModelForCausalLM, prepare_model_for_kbit_training, get_peft_model,PeftModel, PeftConfig
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
torch.cuda.empty_cache()

In [ ]:
# 1=Java-UML
# 2=Python-UML
# 3=Java-OCL
# 4=Python-OCL
What_I_Want=1

In [ ]:
from huggingface_hub import login
token = "YOUR_TOKEN_HERE"
login(token=token)

In [ ]:
DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
# ----------------------------------------------------------------------------
#Resume=True
Resume=False
if Resume==True:
   import wandb
   wandb.init(
      project="huggingface",
      id=####,
      resume="must"
   )
   PathResume=####
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
mistral_checkpoint = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(mistral_checkpoint,
                                          padding_side="left",
                                          add_eos_token=True,
                                          add_bos_token=True,
                                          token=token, use_fast=True, add_prefix_space=True)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "left"
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_use_double_quant=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(mistral_checkpoint,
                                             quantization_config=bnb_config,
                                             device_map='auto',
                                             attn_implementation="flash_attention_2",
                                             use_cache=False
                                             )

model.config.pad_token_id = tokenizer.pad_token_id
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
model.config.use_cache=False
model.config.pretraining_tp=1
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(r=16,lora_alpha=32,
   target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
        "lm_head",
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
def ExtendRopeMistral(model, new_max_seq_len):
   for name, module in model.named_modules():
      if hasattr(module, "rotary_emb") and isinstance(module.rotary_emb, MistralRotaryEmbedding):
         old_dim = module.rotary_emb.dim
         old_base = module.rotary_emb.base

         module.rotary_emb = MistralRotaryEmbedding(
            dim=old_dim,
            max_position_embeddings=new_max_seq_len,
            base=old_base
         ).to(peft_model.device)
         print(f"Extended rotary embedding at {name} to {new_max_seq_len} tokens")
ExtendRopeMistral(peft_model, 32768)

In [ ]:
# ----------------------------------------------------------------------------
def SplitData(Data, TrainSize=.90, TestSize=0.1):
   train_test = Data["train"].train_test_split(test_size=TestSize, shuffle=True, seed=42)
   Temp       = train_test["train"]
   TestData   = train_test["test"]

   train_valid = Temp.train_test_split(test_size=TestSize, shuffle=True, seed=42)
   TrainData   = train_valid["train"]
   ValidData   = train_valid["test"]
   return TrainData, ValidData, TestData
# ----------------------------------------------------------------------------
def GenerateTrainingPrompt(Sample, Instruction):
   if What_I_Want==1 or What_I_Want==2:
      Prompt="""<s>[INST] Below is an instruction that describes a task, paired with an input that provides further context. Write a response, which is in JSON format that appropriately solves the following Task:""".strip()
   else:
      Prompt="""<s>[INST] Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately solves the following Task:""".strip()
   return f"""{Prompt} \n
### Instruction:
{Instruction}
### Input:
{Sample["input"]}
[/INST]

### Response:
{Sample["output"]} </s>"""
# ----------------------------------------------------------------------------
def GenerateText(Sample):
   if What_I_Want==1:
      Instruction="""Generate a concise UML class diagram for the provided Java code. The output should:
1. Define each class and interface only once, including its attributes, methods, and relationships.
2. Include all relationships (inheritance, realization, dependency, association, composition, aggregation) without duplication.
3. Avoid redundant or repeated operations, classes, or relationships.
"""
   elif What_I_Want==2:
      Instruction="""Generate a concise UML class diagram for the provided Python code. The output should:
1. Define each class and interface only once, including its attributes, methods, and relationships.
2. Include all relationships (inheritance, realization, dependency, association, composition, aggregation) without duplication.
3. Avoid redundant or repeated operations, classes, or relationships.
"""
   elif What_I_Want==3:
      Instruction="""Generate an Object Constraint Language (OCL) specification for the provided Java code. The output should:
1. Ensure no repeated or redundant operations or classes.
2. Include only the OCL code for the provided Java code.
3. Do not include statements for items not found in the Java code.
"""
   else:
      Instruction="""Generate an Object Constraint Language (OCL) specification for the provided Python code. The output should:
1. Ensure no repeated or redundant operations or classes.
2. Include only the OCL code for the provided Python code.
3. Do not include statements for items not found in the Python code.
"""
   return {
      "instruction": Instruction,
      "input":  Sample["input"],
      "output": Sample["output"],
      "text": GenerateTrainingPrompt(Sample, Instruction),
 }
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
if What_I_Want==1:
   SavingDirectory = ####
   OutputDirectory=####
elif What_I_Want==2:
   SavingDirectory = ####
   OutputDirectory=####
elif What_I_Want==3:
   SavingDirectory = ####
   OutputDirectory=####
else:
   SavingDirectory = ####
   OutputDirectory=####
# ----------------------------------------------------------------------------
if Resume==False:
   if What_I_Want==1:
      FileName = "JavaUML.json"
   elif What_I_Want==2:
      FileName = "PythonUML.json"
   elif What_I_Want==3:
      FileName = "JavaOCL.json"
   else:
      FileName = "PythonOCL.json"

   Data = load_dataset("json", data_files=FileName)
   Data=Data.map(GenerateText)
   train_dataset = Data['train']
   TrainData, ValidData, TestData = SplitData(Data)

   print(f'Number of prompts: {len(TrainData)}')
   print(f'Column names are: {TrainData.column_names}')

   TrainData.shape
   print(TestData['text'][0])

   TrainData = TrainData.map(lambda x: {'text': x['text']})
   TrainData = TrainData.remove_columns(['program', 'input', 'output', 'instruction'])

   ValidData = ValidData.map(lambda x: {'text': x['text']})
   ValidData = ValidData.remove_columns(['program', 'input', 'output', 'instruction'])

   TestData = TestData.map(lambda x: {'text': x['text']})
   TestData = TestData.remove_columns(['program', 'input', 'output', 'instruction'])

   TrainData.save_to_disk(SavingDirectory+"/TrainData")
   ValidData.save_to_disk(SavingDirectory+"/ValidData")
   TestData.save_to_disk(SavingDirectory+"/TestData")
else:
   from datasets import load_from_disk
   TrainData = load_from_disk(SavingDirectory+"/TrainData")
   ValidData = load_from_disk(SavingDirectory+"/ValidData")
   TestData = load_from_disk(SavingDirectory+"/TestData")
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
training_arguments = TrainingArguments(
   output_dir=OutputDirectory,
   per_device_train_batch_size=1,
   gradient_accumulation_steps= 4,
   optim="adamw_hf",
   learning_rate= 5e-5,
   lr_scheduler_type="cosine", #"linear",
   evaluation_strategy="epoch",
   logging_steps=50,
   num_train_epochs=10,
   max_grad_norm=0.5,
   warmup_ratio=0.2,
   save_strategy="epoch",
   save_total_limit= 10,
   fp16=True,
   do_eval=True,
   eval_steps=50,
   use_cpu=False,
   report_to="wandb"
)

trainer = SFTTrainer(
        model=peft_model,
        train_dataset=TrainData,
        eval_dataset=ValidData,
        peft_config=peft_config,
        dataset_text_field="text",
        args=training_arguments,
        tokenizer=tokenizer,
        packing=False,
		max_seq_length= 8192,
        dataset_kwargs={
           "add_special_tokens": False,
           "append_concat_token": False,
        },
)
checkpoint = training_arguments.resume_from_checkpoint
print("Starting training")

if Resume:
   trainer.train(resume_from_checkpoint=PathResume)
else:
   trainer.train(resume_from_checkpoint=checkpoint)
# ----------------------------------------------------------------------------